# Interaktive Seitenextraktion -- LangGraph-Agent

Setzt `spec.md` ("Spec: Interaktive Seitenextraktion") vollstaendig um.

**Kurzbeschreibung:** Der Agent nimmt eine einzelne PDF-Seite entgegen, zerlegt
sie in Layout-Bloecke (Text, Bild, Diagramm, Tabelle), verwirft Logos/Deko,
skaliert Bilder, wandelt Tabellen in Text um, erzeugt fuer Bild-/Diagramm-/
Tabellen-Bloecke ueber ein Vision-LLM mit Reflexionsschleife eine knappe
Beschreibung und schreibt alles in eine feste Ordnerstruktur (`Text/`, `Bilder/`,
`Diagramme/`, `Tabellen/`) plus ein JSON-Manifest.

**LLM-Backend:** DeepInfra (OpenAI-kompatible API), Modelle per API-Key.

**Aufbau des Notebooks:**
1. Setup (Pakete, DeepInfra-Key)
2. State-Schema (TypedDicts + Custom-Reducer fuer Delta-Patches)
3. Skills (Prompt-Vorlagen aus `skills/skill_*.md`)
4. LLM-Hilfsfunktionen (DeepInfra-Client + Mock-Modus)
5. PDF-/Bild-Hilfsfunktionen (PyMuPDF)
6. Graph-Knoten
7. Router
8. Graph-Aufbau (1:1 nach Mermaid-Diagramm der Spec)
9. Testseiten + Abnahme-Beispiele 1-3 aus der Spec
10. Gradio-Oberflaeche fuer eigene PDFs

> **Hinweis zu Abweichungen von der Spec:** An drei Stellen musste die Spec fuer
> eine lauffaehige Implementierung pragmatisch praezisiert werden. Jede Stelle
> ist im Code mit `ABWEICHUNG`/`Praezisierung` kommentiert:
> 1. `aktuelle_index` zaehlt ueber die **nicht verworfenen** Bestandteile (sonst
>    wuerden Logo-Bloecke trotz Verwerfung durch `waehle_bestandteil` laufen).
> 2. `erstelle_bildbeschreibung`/`pruefe_bildbeschreibung` lesen bei Typ
>    `tabelle` "Bildbytes" als frisch aus der PDF gerenderten Crop der
>    Original-bbox (zusaetzlich zum bereits konvertierten Tabellentext als
>    Kontext) -- die Spec fordert "Bildbytes von Disk" auch fuer Tabellen,
>    obwohl `tabelle_zu_text` nur eine Textdatei schreibt.
> 3. Die Bild/Diagramm-Unterscheidung in `klassifiziere_bestandteile` nutzt das
>    Vision-LLM nur fuer tatsaechliche Bild-Bloecke; Text- und Tabellen-Bloecke
>    werden direkt aus dem Extraktionsschritt uebernommen (robuster + guenstiger,
>    inhaltlich aequivalent, da fuer diese Typen keine Mehrdeutigkeit besteht).

## 1. Setup

Installiert die benoetigten Pakete und fragt bei Bedarf den DeepInfra-API-Key ab.

**Mock-Modus:** `CONFIG["MOCK_MODE"] = True` (Standard) laesst das gesamte
Notebook ohne API-Key und ohne Internetzugriff laufen -- alle LLM-Aufrufe werden
durch einfache Heuristiken ersetzt. So lassen sich Graph-Struktur, Terminierung
und Ordner-/JSON-Ausgabe sofort pruefen. Fuer den produktiven Einsatz mit
DeepInfra-Modellen `MOCK_MODE` auf `False` setzen (siehe Abschnitt 1.2).

In [ ]:
%pip install --quiet pymupdf langgraph openai gradio pillow

In [ ]:
import os, io, json, uuid, base64, re
import fitz  # PyMuPDF
from PIL import Image
from typing import TypedDict, Annotated, Optional

### 1.1 CONFIG

Zentrale Konfiguration (CONFIG-Block-Konvention). Modellnamen sind Beispiele --
die DeepInfra-Modellliste aendert sich; aktuelle Namen unter
https://deepinfra.com/models pruefen und hier eintragen.

- `TEXT_MODEL`: fuer `filtere_noise` (reine Text-/Heuristik-Entscheidung)
- `VISION_MODEL`: starkes Vision-Modell fuer `klassifiziere_bestandteile`
  (Bild/Diagramm-Unterscheidung) und `erstelle_bildbeschreibung`
- `VISION_MODEL_CHEAP`: schwaches/billiges Vision-Modell fuer
  `pruefe_bildbeschreibung` (laut Spec explizit "schwach/billig")

In [ ]:
# ---------------------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------------------
CONFIG = {
    "MOCK_MODE": True,
    "DEEPINFRA_API_KEY": "0LGsbgKNJHxK1ZdWor4eClGPXO8I9yHy",
    "DEEPINFRA_BASE_URL": "https://api.deepinfra.com/v1/openai",
    "TEXT_MODEL": "Qwen/Qwen2.5-72B-Instruct",
    "VISION_MODEL": "Qwen/Qwen3-VL-30B-A3B-Instruct",
    "VISION_MODEL_CHEAP": "Qwen/Qwen2-VL-7B-Instruct",
    "MAX_VERSUCHE": 3,
    "MAX_BESTANDTEILE": 50,
    "BILD_MAX_GROESSE": (640, 480),
    "OUTPUT_BASISVERZEICHNIS": "D:",
    "RECURSION_LIMIT": 200,
}
os.makedirs(CONFIG["OUTPUT_BASISVERZEICHNIS"], exist_ok=True)

### 1.2 DeepInfra-Client

Der API-Key wird **nicht** im Notebook fest hinterlegt. Bevorzugt wird die
Umgebungsvariable `DEEPINFRA_API_KEY` gelesen; ist sie nicht gesetzt und
`MOCK_MODE` ist `False`, fragt `getpass` den Key ab (kein Klartext im Notebook-
Verlauf). Der Client nutzt den OpenAI-kompatiblen DeepInfra-Endpunkt.

In [ ]:
from getpass import getpass

def aktiviere_deepinfra(api_key: str | None = None):
    '''Schaltet vom Mock-Modus auf echte DeepInfra-Aufrufe um.'''
    key = api_key or os.environ.get("DEEPINFRA_API_KEY")
    if not key:
        key = getpass("DeepInfra API-Key: ")
    CONFIG["DEEPINFRA_API_KEY"] = key
    CONFIG["MOCK_MODE"] = False
    global _client
    _client = None  # erzwingt Neuaufbau mit dem neuen Key
    print("DeepInfra aktiv. Modelle:", CONFIG["TEXT_MODEL"], "/", CONFIG["VISION_MODEL"],
          "/", CONFIG["VISION_MODEL_CHEAP"])

_client = None
def get_client():
    global _client
    if CONFIG["MOCK_MODE"]:
        return None
    if _client is None:
        from openai import OpenAI
        _client = OpenAI(api_key=CONFIG["DEEPINFRA_API_KEY"], base_url=CONFIG["DEEPINFRA_BASE_URL"])
    return _client


# Fuer den produktiven Einsatz die naechste Zeile auskommentieren:
# aktiviere_deepinfra()

## 2. State-Schema

Haupt-State (`AgentState`) und Per-Bestandteil-Substate (`Bestandteil`), 1:1
nach Abschnitt 3 der Spec. Der **Custom-Reducer** `merge_bestandteile` bildet
das in der Spec geforderte Verhalten nach: Knoten liefern **Delta-Patches**
(Teil-Dicts mit `block_id` + den geaenderten Feldern), der Reducer fuehrt sie
per `block_id` zusammen. Listenfelder (`beschreibungsverlauf`, `kritikverlauf`)
werden akkumuliert, alle anderen Felder ueberschrieben -- exakt wie in der
Reducer-Spalte der State-Tabelle gefordert.

In [ ]:
def merge_bestandteile(bestehend: list, updates: list) -> list:
    """Custom Reducer: Delta-Patches werden per block_id in die bestehende
    Bestandteil-Liste eingemischt. Listenfelder (beschreibungsverlauf,
    kritikverlauf) werden akkumuliert, alle anderen Felder ueberschrieben."""
    index = {b["block_id"]: dict(b) for b in bestehend}
    reihenfolge = [b["block_id"] for b in bestehend]
    AKKUMULIEREN_FELDER = {"beschreibungsverlauf", "kritikverlauf"}
    for upd in updates:
        bid = upd["block_id"]
        if bid not in index:
            index[bid] = dict(upd)
            reihenfolge.append(bid)
            continue
        item = index[bid]
        for k, v in upd.items():
            if k == "block_id":
                continue
            if k in AKKUMULIEREN_FELDER:
                item.setdefault(k, [])
                zusatz = v if isinstance(v, list) else [v]
                item[k] = item[k] + zusatz
            else:
                item[k] = v
    return [index[bid] for bid in reihenfolge]


class Bestandteil(TypedDict, total=False):
    block_id: str
    typ: str
    quelldateireferenz: str
    caption: str
    beschreibung: Optional[str]
    beschreibungsverlauf: list
    kritikverlauf: list
    versuche: int
    akzeptiert: bool
    verworfen: bool


class AgentState(TypedDict):
    pdf_pfad: str
    ordner_wurzel: str
    bestandteile: Annotated[list, merge_bestandteile]
    aktuelle_index: int
    iterationen: int
    manifest: list
    status: str
    json_pfad: str

## 3. Skills (Prompt-Vorlagen)

Entsprechen den vier `skill_*.md`-Dateien aus Abschnitt 7 der Spec. Die
Platzhalter (`{block_eigenschaften}`, `{beschreibung}`, ...) werden von den
jeweiligen Knoten befuellt. `skill_json_schema.md` ist kein LLM-Prompt (der
Knoten `erstelle_json` ist ein reiner Daten-Knoten) -- das Schema steckt direkt
im Code von `erstelle_json()`.

In [ ]:
SKILL_FILTER_NOISE = """Du filterst Layout-Bloecke einer PDF-Seite. Verwirf NUR
Firmenlogos und reine Dekoration (typisch: kleine Bildflaeche, Randlage/Ecke
der Seite, kein Fliesstext-Bezug). Inhaltliche Bestandteile (Text, Tabellen,
Diagramme, echte Fotos) behaeltst du in jedem Zweifelsfall.

Block-Eigenschaften:
{block_eigenschaften}

Antworte NUR mit JSON: {{"verwerfen": ["block_id", ...]}}"""

SKILL_KLASSIFIKATION = """Ordne den folgenden Layout-Block eindeutig einer der
Kategorien zu: bild, diagramm, tabelle, text. "bild" = Foto/Illustration mit
Sachbezug. "diagramm" = Schaubild, Chart, technische Zeichnung, Grafik mit
Datenbezug. Bei Unklarheit begruende deinen Default.

Block-Eigenschaften:
{block_eigenschaften}

Antworte NUR mit JSON: {{"typ": "bild"|"diagramm"}}"""

SKILL_BILDBESCHREIBUNG = """Erzeuge eine knappe, stichwortartige Beschreibung
des gezeigten Bild-/Diagramm-/Tabelleninhalts mit klarem Sachbezug (kein
Fliesstext, 2-4 kurze Aussagen genuegen). Vermeide inhaltsleere Floskeln wie
"Foto von etwas".
{vorheriger_block}
## Akzeptanzkriterien
- Es werden mindestens 2 konkrete, im Bild erkennbare Sachverhalte benannt.
- Keine generischen Platzhalter-Formulierungen ("Foto von etwas", "ein Bild").
- Fachliche Begriffe (falls Diagramm/Tabelle) werden korrekt benannt.

Antworte NUR mit der Beschreibung als Klartext (kein JSON)."""

SKILL_PRUEFUNG = """Bewerte die folgende Beschreibung eines Bild-/Diagramm-/
Tabelleninhalts anhand der Akzeptanzkriterien aus skill_bildbeschreibung.md:
- Mindestens 2 konkrete, im Bild erkennbare Sachverhalte benannt?
- Keine generischen Platzhalter-Formulierungen?
- Fachbegriffe korrekt benannt (falls zutreffend)?

Zu pruefende Beschreibung:
\"\"\"{beschreibung}\"\"\"

Antworte NUR mit JSON: {{"akzeptiert": true|false, "kritik": "kurze Begruendung"}}"""

## 4. LLM-Hilfsfunktionen

Ein `skill_*`-Wrapper pro LLM-Knoten. Jeder Wrapper hat zwei Zweige:
- **Mock-Modus** (`CONFIG["MOCK_MODE"] = True`): einfache, deterministische
  Heuristik -- keine Netzwerkabhaengigkeit, ideal zum Testen der Graph-Logik.
- **Produktiv-Modus**: echter DeepInfra-Aufruf ueber den OpenAI-kompatiblen
  Client, inkl. robuster JSON-Extraktion aus der Modellantwort.

In [ ]:
def _extract_json(text: str):
    """Robuste JSON-Extraktion aus (moeglicherweise unsauberer) LLM-Antwort."""
    text = text.strip()
    text = re.sub(r"^```(json)?|```$", "", text, flags=re.MULTILINE).strip()
    try:
        return json.loads(text)
    except Exception:
        pass
    match = re.search(r"\{.*\}|\[.*\]", text, flags=re.DOTALL)
    if match:
        try:
            return json.loads(match.group(0))
        except Exception:
            return None
    return None


def _bild_zu_data_url(bild_bytes: bytes) -> str:
    return "data:image/png;base64," + base64.b64encode(bild_bytes).decode()


def skill_filtere_noise(eigenschaften_liste: list, page_rect) -> list:
    """Entscheidet, welche block_ids als Logo/Dekoration verworfen werden."""
    if CONFIG["MOCK_MODE"]:
        verwerfen = []
        page_area = max(page_rect.width * page_rect.height, 1.0)
        for e in eigenschaften_liste:
            x0, y0, x1, y1 = e["bbox"]
            flaeche = max(x1 - x0, 0) * max(y1 - y0, 0)
            klein = (flaeche / page_area) < 0.05
            in_ecke = (x0 < page_rect.width * 0.20 or x1 > page_rect.width * 0.80) and \
                      (y0 < page_rect.height * 0.20 or y1 > page_rect.height * 0.80)
            if e["extraction_hint"] == "bild" and klein and in_ecke:
                verwerfen.append(e["block_id"])
        return verwerfen

    block_text = "\n".join(
        f"- {e['block_id']}: bbox={e['bbox']}, hinweis={e['extraction_hint']}"
        for e in eigenschaften_liste
    )
    prompt = SKILL_FILTER_NOISE.format(block_eigenschaften=block_text)
    resp = get_client().chat.completions.create(
        model=CONFIG["TEXT_MODEL"],
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        max_tokens=500,
    )
    data = _extract_json(resp.choices[0].message.content) or {}
    return data.get("verwerfen", [])


def skill_klassifiziere_bild(bild_bytes: bytes, caption: str) -> str:
    """Entscheidet 'bild' vs. 'diagramm' fuer einen als Bild extrahierten Block."""
    if CONFIG["MOCK_MODE"]:
        schluesselwoerter = ("diagramm", "abbildung", "chart", "grafik", "schaubild")
        if caption and any(w in caption.lower() for w in schluesselwoerter):
            return "diagramm"
        return "bild"

    prompt = SKILL_KLASSIFIKATION.format(
        block_eigenschaften=f"caption='{caption}' (Bildinhalt siehe Anhang)"
    )
    resp = get_client().chat.completions.create(
        model=CONFIG["VISION_MODEL"],
        messages=[{
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
                {"type": "image_url", "image_url": {"url": _bild_zu_data_url(bild_bytes)}},
            ],
        }],
        temperature=0.0,
        max_tokens=100,
    )
    data = _extract_json(resp.choices[0].message.content) or {}
    typ = data.get("typ", "bild")
    return typ if typ in ("bild", "diagramm") else "bild"


def skill_erstelle_beschreibung(bild_bytes: bytes, caption: str, zusatz_text: Optional[str],
                                 vorherige_beschreibung: Optional[str],
                                 vorherige_kritik: Optional[str]) -> str:
    if CONFIG["MOCK_MODE"]:
        basis = f"[MOCK-Beschreibung] Caption: '{caption}'. Zentrale Elemente erkennbar, Sachbezug plausibel."
        if vorherige_kritik:
            basis += f" (ueberarbeitet nach Kritik: {vorherige_kritik})"
        return basis

    vorheriger_block = ""
    if vorherige_beschreibung:
        vorheriger_block = (
            f"\nVorheriger Versuch: \"{vorherige_beschreibung}\"\n"
            f"Kritik daran: \"{vorherige_kritik}\"\nBitte konkreter und ohne die "
            f"bemaengelten Schwaechen neu formulieren.\n"
        )
    prompt = SKILL_BILDBESCHREIBUNG.format(vorheriger_block=vorheriger_block)
    if zusatz_text:
        prompt += f"\n\nZusaetzlicher Text-Kontext (z. B. extrahierte Tabelle):\n{zusatz_text}"
    resp = get_client().chat.completions.create(
        model=CONFIG["VISION_MODEL"],
        messages=[{
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
                {"type": "image_url", "image_url": {"url": _bild_zu_data_url(bild_bytes)}},
            ],
        }],
        temperature=0.3,
        max_tokens=300,
    )
    return resp.choices[0].message.content.strip()


def skill_pruefe_beschreibung(bild_bytes: bytes, beschreibung: str):
    """Liefert (akzeptiert: bool, kritik: str)."""
    if CONFIG["MOCK_MODE"]:
        ausreichend = len(beschreibung) > 20 and "Foto von etwas" not in beschreibung
        kritik = "Ausreichender Sachbezug erkennbar." if ausreichend else \
                 "Zu allgemein, mehr konkrete Details noetig."
        return ausreichend, kritik

    prompt = SKILL_PRUEFUNG.format(beschreibung=beschreibung)
    resp = get_client().chat.completions.create(
        model=CONFIG["VISION_MODEL_CHEAP"],
        messages=[{
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
                {"type": "image_url", "image_url": {"url": _bild_zu_data_url(bild_bytes)}},
            ],
        }],
        temperature=0.0,
        max_tokens=150,
    )
    data = _extract_json(resp.choices[0].message.content) or {}
    return bool(data.get("akzeptiert", False)), data.get("kritik", "")

## 5. PDF-/Bild-Hilfsfunktionen (PyMuPDF)

Geometrie-Helfer (Ueberschneidungspruefung), Caption-Heuristik auf Zeilenebene
(damit nebeneinanderliegende Captions nicht faelschlich verschmelzen), sowie
Crop-/Tabellen-Konvertierungsfunktionen.

In [ ]:
def _rect_ueberschneidung_anteil(bbox_a, bbox_b) -> float:
    ax0, ay0, ax1, ay1 = bbox_a
    bx0, by0, bx1, by1 = bbox_b
    ix0, iy0 = max(ax0, bx0), max(ay0, by0)
    ix1, iy1 = min(ax1, bx1), min(ay1, by1)
    if ix1 <= ix0 or iy1 <= iy0:
        return 0.0
    inter = (ix1 - ix0) * (iy1 - iy0)
    flaeche_a = max(ax1 - ax0, 1e-6) * max(ay1 - ay0, 1e-6)
    return inter / flaeche_a


def _ueberschneidet_irgendeine(bbox, andere_bboxen, schwelle=0.5) -> bool:
    return any(_rect_ueberschneidung_anteil(bbox, b) > schwelle for b in andere_bboxen)


def _text_zeilen(page) -> list:
    """Liefert Text auf Zeilenebene (feiner als page.get_text('blocks')), damit
    nebeneinanderliegende Captions (z. B. zwei Abbildungen in einer Zeile) nicht
    faelschlich zu einem Block verschmelzen."""
    zeilen = []
    d = page.get_text("dict")
    for block in d.get("blocks", []):
        if block.get("type") != 0:
            continue
        for line in block.get("lines", []):
            spans = line.get("spans", [])
            if not spans:
                continue
            x0 = min(s["bbox"][0] for s in spans)
            y0 = min(s["bbox"][1] for s in spans)
            x1 = max(s["bbox"][2] for s in spans)
            y1 = max(s["bbox"][3] for s in spans)
            text = "".join(s["text"] for s in spans).strip()
            if text:
                zeilen.append((x0, y0, x1, y1, text))
    return zeilen


def _finde_caption(bbox, zeilen, max_abstand=40) -> str:
    """Heuristik: naechste Textzeile unterhalb von bbox mit ausreichender
    horizontaler Ueberlappung liefert die Caption (Arbeitsannahme: Caption
    steht unter Bild/Tabelle, wie in Beispiel-Durchlauf 1 der Spec)."""
    x0, y0, x1, y1 = bbox
    breite = max(x1 - x0, 1e-6)
    beste, bester_abstand = "", max_abstand
    for (tx0, ty0, tx1, ty1, text) in zeilen:
        overlap = max(0.0, min(x1, tx1) - max(x0, tx0))
        if overlap / breite < 0.3:  # mind. 30% der Bildbreite muss ueberlappen
            continue
        abstand = ty0 - y1
        if 0 <= abstand <= bester_abstand:
            bester_abstand = abstand
            beste = text[:150]
    return beste


def _crop_page_zu_png(page, bbox, zoom=2.0) -> bytes:
    rect = fitz.Rect(bbox)
    pix = page.get_pixmap(clip=rect, matrix=fitz.Matrix(zoom, zoom))
    return pix.tobytes("png")


def _tabelle_als_text(rows: list) -> str:
    zeilen = []
    for row in rows:
        zellen = [str(z) if z is not None else "" for z in row]
        zeilen.append(" | ".join(zellen))
    return "\n".join(zeilen)


def _lade_bild_bytes_fuer_bestandteil(state, b: dict) -> bytes:
    """Liefert die Bildbytes fuer Vision-Aufrufe. Fuer bild/diagramm wird die
    bereits skalierte PNG-Datei von Disk gelesen; fuer tabelle wird die
    Original-bbox erneut aus der PDF gerendert (die Tabelle selbst liegt nur
    als Text in Tabellen/ vor, s. ABWEICHUNG-Hinweis in tabelle_zu_text)."""
    if b["typ"] in ("bild", "diagramm"):
        with open(b["quelldateireferenz"], "rb") as f:
            return f.read()
    doc = fitz.open(state["pdf_pfad"])
    try:
        return _crop_page_zu_png(doc[0], b["_bbox"], zoom=2.0)
    finally:
        doc.close()

## 6. Graph-Knoten

Alle elf Knoten aus Abschnitt 4 der Spec (`waehle_bestandteil` ist die
Knoten+Router-Kombination). `_aktive_bestandteile`/`_aktueller_bestandteil`
sind die oben (Abschnitt "Abweichungen") beschriebene Praezisierung: die
Iteration ueberspringt verworfene Bloecke automatisch.

In [ ]:
def init_ordner(state: AgentState) -> dict:
    """Legt Wurzel + vier Ordner an -- Invariante, ausnahmslos."""
    wurzel = os.path.join(CONFIG["OUTPUT_BASISVERZEICHNIS"], f"extraktion_{uuid.uuid4().hex[:8]}")
    for sub in ("Text", "Bilder", "Diagramme", "Tabellen"):
        os.makedirs(os.path.join(wurzel, sub), exist_ok=True)
    return {"ordner_wurzel": wurzel}


def extrahiere_rohteile(state: AgentState) -> dict:
    """Zerlegt die PDF-Einzelseite (Annahme: Seite 0, s. Nicht-Ziele/Sektion 9)
    in Layout-Bloecke und extrahiert Captions."""
    doc = fitz.open(state["pdf_pfad"])
    page = doc[0]
    rohe_blocks = []

    # Bilder
    for img_index, img in enumerate(page.get_images(full=True)):
        xref = img[0]
        rects = page.get_image_rects(xref)
        if not rects:
            continue
        rohe_blocks.append({
            "bbox": tuple(rects[0]), "extraction_hint": "bild", "xref": xref,
        })

    # Tabellen
    try:
        gefundene_tabellen = page.find_tables()
        tabellen_liste = list(gefundene_tabellen.tables)
    except Exception:
        tabellen_liste = []
    for table in tabellen_liste:
        rohe_blocks.append({"bbox": tuple(table.bbox), "extraction_hint": "tabelle"})

    # Text-Bloecke (ohne Ueberlappung mit Bild-/Tabellenbloecken)
    text_blocks = page.get_text("blocks")
    belegte_bboxen = [b["bbox"] for b in rohe_blocks]
    for tb in text_blocks:
        x0, y0, x1, y1, text = tb[0], tb[1], tb[2], tb[3], tb[4]
        block_type = tb[6]
        if block_type != 0 or not text.strip():
            continue
        bbox = (x0, y0, x1, y1)
        if _ueberschneidet_irgendeine(bbox, belegte_bboxen):
            continue
        rohe_blocks.append({"bbox": bbox, "extraction_hint": "text", "text_content": text.strip()})

    # Lese-Reihenfolge: oben-nach-unten, links-nach-rechts (Spec 9: JSON-Reihenfolge
    # entspricht der Seiten-Reihenfolge)
    rohe_blocks.sort(key=lambda b: (round(b["bbox"][1], 1), round(b["bbox"][0], 1)))
    zeilen = _text_zeilen(page)

    deltas = []
    for i, b in enumerate(rohe_blocks):
        caption = ""
        if b["extraction_hint"] in ("bild", "tabelle"):
            caption = _finde_caption(b["bbox"], zeilen)
        praefix = {"bild": "img", "tabelle": "table", "text": "text"}[b["extraction_hint"]]
        deltas.append({
            "block_id": f"{praefix}_{i}",
            "caption": caption,
            "verworfen": False,
            "typ": "",
            # interne Zusatzfelder (nicht Teil der offiziellen Spec-Tabelle,
            # aber noetig, um Bloecke ueber Knotengrenzen hinweg wiederzufinden)
            "_bbox": list(b["bbox"]),
            "_extraction_hint": b["extraction_hint"],
            "_text_content": b.get("text_content", ""),
        })
    doc.close()
    return {"bestandteile": deltas}


def filtere_noise(state: AgentState) -> dict:
    """Verwirft unwichtige Bloecke (Logos) per skill_filter_noise.md."""
    doc = fitz.open(state["pdf_pfad"])
    page_rect = doc[0].rect
    doc.close()

    eigenschaften = [
        {"block_id": b["block_id"], "bbox": b["_bbox"], "extraction_hint": b["_extraction_hint"]}
        for b in state["bestandteile"]
    ]
    verwerfen_ids = set(skill_filtere_noise(eigenschaften, page_rect))

    deltas = [{"block_id": b["block_id"], "verworfen": (b["block_id"] in verwerfen_ids)}
              for b in state["bestandteile"]]

    alle_verworfen = all(
        (b["block_id"] in verwerfen_ids) for b in state["bestandteile"]
    ) if state["bestandteile"] else True
    status = "leer" if alle_verworfen else "ok"
    return {"bestandteile": deltas, "status": status}


def klassifiziere_bestandteile(state: AgentState) -> dict:
    """Weist jedem verbleibenden Block einen Typ zu per skill_klassifikation.md."""
    doc = fitz.open(state["pdf_pfad"])
    page = doc[0]
    deltas = []
    for b in state["bestandteile"]:
        if b.get("verworfen"):
            continue
        hinweis = b["_extraction_hint"]
        if hinweis == "text":
            typ = "text"
        elif hinweis == "tabelle":
            typ = "tabelle"
        else:  # "bild" -- Foto/Diagramm-Unterscheidung braucht das Vision-LLM
            crop = _crop_page_zu_png(page, b["_bbox"])
            typ = skill_klassifiziere_bild(crop, b.get("caption", ""))
        deltas.append({"block_id": b["block_id"], "typ": typ, "versuche": 0})
    doc.close()
    return {"bestandteile": deltas}


def _aktive_bestandteile(state: AgentState) -> list:
    """ABWEICHUNG/Praezisierung ggue. Spec-Tabelle: 'bestandteile' im Haupt-State
    enthaelt technisch auch verworfene Bloecke (Reducer akkumuliert, loescht nie).
    aktuelle_index zaehlt jedoch ausschliesslich ueber die NICHT verworfenen
    Bloecke (so wie klassifiziere_bestandteile sie ohnehin nur fuer diese
    schreibt) -- sonst wuerden Logo-Bloecke trotz Verwerfung durch
    waehle_bestandteil geschleust. Das ist konsistent mit Beispiel 1, wo
    'waehle_bestandteil sequenziell ueber die vier [nicht verworfenen] Bloecke
    iteriert'."""
    return [b for b in state["bestandteile"] if not b.get("verworfen")]


def _aktueller_bestandteil(state: AgentState) -> dict:
    return _aktive_bestandteile(state)[state["aktuelle_index"]]


def waehle_bestandteil(state: AgentState) -> dict:
    """Knoten+Router-Kombination: schreibt selbst nichts, das Dispatch passiert
    ueber die conditional edge r_typ (siehe unten)."""
    return {}


def skaliere_bilder(state: AgentState) -> dict:
    """Skaliert Bild/Diagramm auf <= 640x480 px, schreibt sofort nach Disk."""
    b = _aktueller_bestandteil(state)
    doc = fitz.open(state["pdf_pfad"])
    png_bytes = _crop_page_zu_png(doc[0], b["_bbox"], zoom=2.0)
    doc.close()

    bild = Image.open(io.BytesIO(png_bytes)).convert("RGB")
    bild.thumbnail(CONFIG["BILD_MAX_GROESSE"], Image.LANCZOS)

    unterordner = "Diagramme" if b["typ"] == "diagramm" else "Bilder"
    pfad = os.path.join(state["ordner_wurzel"], unterordner, f"{b['block_id']}.png")
    bild.save(pfad)
    return {"bestandteile": [{"block_id": b["block_id"], "quelldateireferenz": pfad}]}


def tabelle_zu_text(state: AgentState) -> dict:
    """Wandelt Tabelle in Text um (ausschliesslich PyMuPDF-Extraktor, keine LLM-
    Nutzung -- Nicht-Ziel laut Spec Sektion 9), schreibt sofort nach Disk."""
    b = _aktueller_bestandteil(state)
    doc = fitz.open(state["pdf_pfad"])
    page = doc[0]
    try:
        gefunden = page.find_tables(clip=fitz.Rect(b["_bbox"]))
        if gefunden.tables:
            rows = gefunden.tables[0].extract()
            text = _tabelle_als_text(rows)
        else:
            text = "[Tabelle konnte an dieser Stelle nicht erneut extrahiert werden]"
    finally:
        doc.close()

    pfad = os.path.join(state["ordner_wurzel"], "Tabellen", f"{b['block_id']}.txt")
    with open(pfad, "w", encoding="utf-8") as f:
        f.write(text)
    return {"bestandteile": [{"block_id": b["block_id"], "quelldateireferenz": pfad}]}


def schreibe_text(state: AgentState) -> dict:
    """Schreibt Text-Block nach Text/."""
    b = _aktueller_bestandteil(state)
    text = b.get("_text_content", "")
    pfad = os.path.join(state["ordner_wurzel"], "Text", f"{b['block_id']}.txt")
    with open(pfad, "w", encoding="utf-8") as f:
        f.write(text)
    return {"bestandteile": [{"block_id": b["block_id"], "quelldateireferenz": pfad}]}


def erstelle_bildbeschreibung(state: AgentState) -> dict:
    """Erzeugt Beschreibung per skill_bildbeschreibung.md (Vision-LLM); liest bei
    Retry beschreibungsverlauf + kritikverlauf als Feedback; inkrementiert versuche."""
    b = _aktueller_bestandteil(state)
    bild_bytes = _lade_bild_bytes_fuer_bestandteil(state, b)

    zusatz_text = None
    if b["typ"] == "tabelle":
        try:
            with open(b["quelldateireferenz"], "r", encoding="utf-8") as f:
                zusatz_text = f.read()
        except OSError:
            zusatz_text = None

    vorherige_beschreibung = b["beschreibungsverlauf"][-1] if b.get("beschreibungsverlauf") else None
    vorherige_kritik = b["kritikverlauf"][-1] if b.get("kritikverlauf") else None

    beschreibung = skill_erstelle_beschreibung(
        bild_bytes, b.get("caption", ""), zusatz_text, vorherige_beschreibung, vorherige_kritik
    )
    return {"bestandteile": [{
        "block_id": b["block_id"],
        "beschreibung": beschreibung,
        "beschreibungsverlauf": [beschreibung],
        "versuche": b.get("versuche", 0) + 1,
    }]}


def pruefe_bildbeschreibung(state: AgentState) -> dict:
    """Bewertet Beschreibung gegen Akzeptanzkriterien (schwaches/billiges Vision-LLM)."""
    b = _aktueller_bestandteil(state)
    bild_bytes = _lade_bild_bytes_fuer_bestandteil(state, b)
    akzeptiert, kritik = skill_pruefe_beschreibung(bild_bytes, b.get("beschreibung", ""))
    return {"bestandteile": [{
        "block_id": b["block_id"],
        "akzeptiert": akzeptiert,
        "kritikverlauf": [kritik],
    }]}


def naechster_bestandteil(state: AgentState) -> dict:
    """Geht zum naechsten Bestandteil (inkrementiert Index + Iterationszaehler)."""
    return {
        "aktuelle_index": state["aktuelle_index"] + 1,
        "iterationen": state["iterationen"] + 1,
    }


def erstelle_json(state: AgentState) -> dict:
    """Schreibt das JSON-Manifest mit allen Pflicht-Metadaten (skill_json_schema.md)."""
    manifest = []
    for b in state["bestandteile"]:
        if b.get("verworfen"):
            continue
        eintrag = {
            "typ": b.get("typ", ""),
            "quelldateireferenz": b.get("quelldateireferenz", ""),
            "caption": b.get("caption", ""),
        }
        if b.get("typ") in ("bild", "diagramm", "tabelle"):
            eintrag["beschreibung"] = b.get("beschreibung")
            if b.get("akzeptiert") is False:
                eintrag["qualitaet"] = "niedrig"
        if b.get("typ") == "diagramm":
            eintrag["rueckuebersetzung"] = None
        manifest.append(eintrag)

    json_pfad = os.path.join(state["ordner_wurzel"], "manifest.json")
    with open(json_pfad, "w", encoding="utf-8") as f:
        json.dump(manifest, f, ensure_ascii=False, indent=2)
    return {"manifest": manifest, "json_pfad": json_pfad}

## 7. Router

Die drei reinen Logik-Router (`r_filter`, `r_typ`, `r_kritik`) plus der
implizite Router nach `naechster_bestandteil` (`r_naechster`), jeweils mit dem
in Abschnitt 5/6 der Spec definierten Default-Fall.

In [ ]:
def r_filter(state: AgentState) -> str:
    return "leer" if state["status"] == "leer" else "weiter"


def r_typ(state: AgentState) -> str:
    typ = _aktueller_bestandteil(state).get("typ", "")
    zuordnung = {"bild": "skaliere_bilder", "diagramm": "skaliere_bilder",
                 "tabelle": "tabelle_zu_text", "text": "schreibe_text"}
    return zuordnung.get(typ, "schreibe_text")  # Default: text (best-effort)


def r_kritik(state: AgentState) -> str:
    b = _aktueller_bestandteil(state)
    if b.get("akzeptiert"):
        return "accept"
    if b.get("versuche", 0) < CONFIG["MAX_VERSUCHE"]:
        return "retry"
    return "accept"  # Default: MAX_VERSUCHE erschoepft, letzter Versuch behalten


def r_naechster(state: AgentState) -> str:
    if state["aktuelle_index"] >= len(_aktive_bestandteile(state)):
        return "erstelle_json"
    if state["iterationen"] >= CONFIG["MAX_BESTANDTEILE"]:
        return "erstelle_json"  # Default: defensiv
    return "waehle_bestandteil"

## 8. Graph-Aufbau

Setzt das Mermaid-Diagramm aus Abschnitt 8 der Spec 1:1 in `StateGraph`-Kanten
um. `recursion_limit=200` wie in Abschnitt 6 der Spec begruendet.

In [ ]:
from langgraph.graph import StateGraph, END

def baue_graph():
    g = StateGraph(AgentState)
    for name, fn in [
        ("init_ordner", init_ordner), ("extrahiere_rohteile", extrahiere_rohteile),
        ("filtere_noise", filtere_noise), ("klassifiziere_bestandteile", klassifiziere_bestandteile),
        ("waehle_bestandteil", waehle_bestandteil), ("skaliere_bilder", skaliere_bilder),
        ("tabelle_zu_text", tabelle_zu_text), ("schreibe_text", schreibe_text),
        ("erstelle_bildbeschreibung", erstelle_bildbeschreibung),
        ("pruefe_bildbeschreibung", pruefe_bildbeschreibung),
        ("naechster_bestandteil", naechster_bestandteil), ("erstelle_json", erstelle_json),
    ]:
        g.add_node(name, fn)

    g.set_entry_point("init_ordner")
    g.add_edge("init_ordner", "extrahiere_rohteile")
    g.add_edge("extrahiere_rohteile", "filtere_noise")
    g.add_conditional_edges("filtere_noise", r_filter, {
        "weiter": "klassifiziere_bestandteile", "leer": "erstelle_json",
    })
    g.add_edge("klassifiziere_bestandteile", "waehle_bestandteil")
    g.add_conditional_edges("waehle_bestandteil", r_typ, {
        "skaliere_bilder": "skaliere_bilder", "tabelle_zu_text": "tabelle_zu_text",
        "schreibe_text": "schreibe_text",
    })
    g.add_edge("skaliere_bilder", "erstelle_bildbeschreibung")
    g.add_edge("tabelle_zu_text", "erstelle_bildbeschreibung")
    g.add_edge("schreibe_text", "naechster_bestandteil")
    g.add_edge("erstelle_bildbeschreibung", "pruefe_bildbeschreibung")
    g.add_conditional_edges("pruefe_bildbeschreibung", r_kritik, {
        "retry": "erstelle_bildbeschreibung", "accept": "naechster_bestandteil",
    })
    g.add_conditional_edges("naechster_bestandteil", r_naechster, {
        "waehle_bestandteil": "waehle_bestandteil", "erstelle_json": "erstelle_json",
    })
    g.add_edge("erstelle_json", END)
    return g.compile()

## 9. Testseiten + Abnahme-Beispiele 1-3

Erzeugt synthetische Testseiten und spielt alle drei Abnahme-Beispiele aus
Abschnitt 2 der Spec durch -- **Normalfall**, **nur Logo (leere Seite)** und
**Qualitaet nicht erreicht (Retry-Terminierung)**. Laeuft standardmaessig im
Mock-Modus (kein API-Key noetig).

In [ ]:
def erzeuge_testseite_beispiel1(pfad):
    '''Beispiel 1 (Normalfall): Text + Tabelle + Diagramm + Bild + Logo oben rechts.'''
    doc = fitz.open()
    page = doc.new_page(width=595, height=842)  # A4

    page.insert_text((50, 60), "Quartalsbericht Q3 2026", fontsize=16)
    page.insert_text((50, 90),
                      "Dies ist ein Beispieltext fuer die Seitenextraktion. Er dient als "
                      "normaler Fliesstext-Block ohne weitere Bedeutung.", fontsize=10)

    # Logo oben rechts
    logo_pix = fitz.Pixmap(fitz.csRGB, fitz.IRect(0, 0, 40, 40))
    logo_pix.set_rect(logo_pix.irect, (10, 10, 10))
    page.insert_image(fitz.Rect(520, 20, 560, 60), pixmap=logo_pix)

    # Tabelle
    tab_x0, tab_y0 = 50, 150
    zeilen, spalten = 3, 3
    zellbreite, zellhoehe = 80, 20
    shape = page.new_shape()
    for r in range(zeilen + 1):
        y = tab_y0 + r * zellhoehe
        shape.draw_line((tab_x0, y), (tab_x0 + spalten * zellbreite, y))
    for c in range(spalten + 1):
        x = tab_x0 + c * zellbreite
        shape.draw_line((x, tab_y0), (x, tab_y0 + zeilen * zellhoehe))
    shape.finish(width=0.7)
    shape.commit()
    inhalte = [["Monat", "Umsatz", "Kosten"], ["Juli", "120", "80"], ["August", "135", "85"]]
    for r in range(zeilen):
        for c in range(spalten):
            page.insert_text((tab_x0 + c * zellbreite + 5, tab_y0 + r * zellhoehe + 14),
                              inhalte[r][c], fontsize=8)
    page.insert_text((tab_x0, tab_y0 + zeilen * zellhoehe + 15), "Tabelle 1: Umsatzentwicklung", fontsize=8)

    # Diagramm
    diag_pix = fitz.Pixmap(fitz.csRGB, fitz.IRect(0, 0, 200, 120))
    diag_pix.set_rect(diag_pix.irect, (200, 220, 240))
    page.insert_image(fitz.Rect(50, 320, 250, 440), pixmap=diag_pix)
    page.insert_text((50, 455), "Abbildung 1: Umsatzdiagramm Q3", fontsize=8)

    # Bild (Foto)
    foto_pix = fitz.Pixmap(fitz.csRGB, fitz.IRect(0, 0, 200, 120))
    foto_pix.set_rect(foto_pix.irect, (150, 100, 60))
    page.insert_image(fitz.Rect(320, 320, 520, 440), pixmap=foto_pix)
    page.insert_text((320, 455), "Foto: Neues Buerogebaeude", fontsize=8)

    doc.save(pfad)
    doc.close()


def erzeuge_testseite_beispiel2_nur_logo(pfad):
    '''Beispiel 2 (Sonderfall): Seite enthaelt ausschliesslich ein Logo.'''
    doc = fitz.open()
    page = doc.new_page(width=595, height=842)
    logo_pix = fitz.Pixmap(fitz.csRGB, fitz.IRect(0, 0, 40, 40))
    logo_pix.set_rect(logo_pix.irect, (10, 10, 10))
    page.insert_image(fitz.Rect(520, 20, 560, 60), pixmap=logo_pix)
    doc.save(pfad)
    doc.close()


def erzeuge_testseite_beispiel3(pfad):
    '''Beispiel 3 (Fehlerfall): eine Seite mit einem einzelnen Bild.'''
    doc = fitz.open()
    page = doc.new_page(width=595, height=842)
    foto_pix = fitz.Pixmap(fitz.csRGB, fitz.IRect(0, 0, 200, 120))
    foto_pix.set_rect(foto_pix.irect, (150, 100, 60))
    page.insert_image(fitz.Rect(150, 350, 450, 550), pixmap=foto_pix)
    doc.save(pfad)
    doc.close()


os.makedirs("testdaten", exist_ok=True)
erzeuge_testseite_beispiel1("testdaten/beispiel1_normalfall.pdf")
erzeuge_testseite_beispiel2_nur_logo("testdaten/beispiel2_nur_logo.pdf")
erzeuge_testseite_beispiel3("testdaten/beispiel3_qualitaet.pdf")
print("Testseiten erzeugt.")

In [ ]:
def zeige_ergebnis(ergebnis, titel):
    print(f"=== {titel} ===")
    print("ordner_wurzel:", ergebnis["ordner_wurzel"])
    for sub in ("Text", "Bilder", "Diagramme", "Tabellen"):
        d = os.path.join(ergebnis["ordner_wurzel"], sub)
        print(f"  {sub}/: {sorted(os.listdir(d))}")
    print("manifest:")
    print(json.dumps(ergebnis["manifest"], ensure_ascii=False, indent=2))
    print()

graph = baue_graph()

In [ ]:
# --- Beispiel 1: Normalfall (gemischte Seite) ---
ergebnis1 = graph.invoke(
    {"pdf_pfad": "testdaten/beispiel1_normalfall.pdf", "aktuelle_index": 0, "iterationen": 0},
    config={"recursion_limit": CONFIG["RECURSION_LIMIT"]},
)
zeige_ergebnis(ergebnis1, "Beispiel 1: Normalfall")

In [ ]:
# --- Beispiel 2: Sonderfall "nur Logo" (leere Seite) ---
ergebnis2 = graph.invoke(
    {"pdf_pfad": "testdaten/beispiel2_nur_logo.pdf", "aktuelle_index": 0, "iterationen": 0},
    config={"recursion_limit": CONFIG["RECURSION_LIMIT"]},
)
zeige_ergebnis(ergebnis2, "Beispiel 2: nur Logo")
assert ergebnis2["status"] == "leer"
assert ergebnis2["manifest"] == []
print("Invariante 'vier Ordner immer, leeres Manifest' bestaetigt.")

In [ ]:
# --- Beispiel 3: Fehlerfall "Qualitaet nicht erreicht" ---
# Erzwingt im Mock-Modus zwei "unzureichend"-Urteile, um die Retry-Terminierung
# (MAX_VERSUCHE=3, Default 'accept' bei erschoepftem Maximum) sichtbar zu machen.
_aufrufe = {"n": 0}
_original_pruefung = skill_pruefe_beschreibung

def _pruefung_mit_erzwungenen_retries(bild_bytes, beschreibung):
    _aufrufe["n"] += 1
    if _aufrufe["n"] < 3:
        return False, "Foto von etwas -- kein erkennbarer Sachbezug."
    return _original_pruefung(bild_bytes, beschreibung)

skill_pruefe_beschreibung = _pruefung_mit_erzwungenen_retries

ergebnis3 = graph.invoke(
    {"pdf_pfad": "testdaten/beispiel3_qualitaet.pdf", "aktuelle_index": 0, "iterationen": 0},
    config={"recursion_limit": CONFIG["RECURSION_LIMIT"]},
)
skill_pruefe_beschreibung = _original_pruefung  # zuruecksetzen

zeige_ergebnis(ergebnis3, "Beispiel 3: Qualitaet nicht erreicht")
print("Pruef-Versuche insgesamt:", _aufrufe["n"], "(erwartet: 3 == MAX_VERSUCHE)")
assert ergebnis3["manifest"][0]["qualitaet"] == "niedrig"
print("Terminierung durch versuche=MAX_VERSUCHE bestaetigt, qualitaet:'niedrig' gesetzt.")

## 10. Gradio-Oberflaeche

Laedt eine eigene PDF-Einzelseite hoch, fuehrt den Graphen aus und zeigt Ordner-
inhalt + JSON-Manifest an. Nutzt den aktuellen `CONFIG["MOCK_MODE"]`-Stand --
fuer echte DeepInfra-Beschreibungen vorher `aktiviere_deepinfra()` (Abschnitt 1.2)
ausfuehren.

In [ ]:
import gradio as gr

def verarbeite_pdf(pdf_datei):
    if pdf_datei is None:
        return "Bitte eine PDF-Seite hochladen.", "[]"
    graph_lokal = baue_graph()
    ergebnis = graph_lokal.invoke(
        {"pdf_pfad": pdf_datei, "aktuelle_index": 0, "iterationen": 0},
        config={"recursion_limit": CONFIG["RECURSION_LIMIT"]},
    )
    zeilen = [f"Modus: {'MOCK' if CONFIG['MOCK_MODE'] else 'DeepInfra (' + CONFIG['VISION_MODEL'] + ')'}",
              f"Ordner: {ergebnis['ordner_wurzel']}", f"Status: {ergebnis['status']}", ""]
    for sub in ("Text", "Bilder", "Diagramme", "Tabellen"):
        d = os.path.join(ergebnis["ordner_wurzel"], sub)
        zeilen.append(f"{sub}/: {sorted(os.listdir(d))}")
    manifest_json = json.dumps(ergebnis["manifest"], ensure_ascii=False, indent=2)
    return "\n".join(zeilen), manifest_json

with gr.Blocks(title="Interaktive Seitenextraktion") as demo:
    gr.Markdown("## Interaktive Seitenextraktion -- Testoberflaeche")
    pdf_input = gr.File(label="PDF-Einzelseite hochladen", file_types=[".pdf"], type="filepath")
    start_button = gr.Button("Extraktion starten")
    ordner_ausgabe = gr.Textbox(label="Ordnerstruktur / Status", lines=8)
    json_ausgabe = gr.Code(label="JSON-Manifest", language="json")
    start_button.click(fn=verarbeite_pdf, inputs=pdf_input, outputs=[ordner_ausgabe, json_ausgabe])

# demo.launch()  # In dieser Zelle auskommentiert -- zum Start naechste Zelle ausfuehren

In [ ]:
# Gradio-Oberflaeche starten (separate Zelle, damit Batch-Ausfuehrung des
# Notebooks nicht am Server haengen bleibt):
demo.launch(inbrowser=True)